In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/q19_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
SHIPMODES = ["AIR", "AIRREG"]

# Precompute lineitem columns to cut down __getitem__ calls
lq_line = lineitem["L_QUANTITY"]
si_line = lineitem["L_SHIPINSTRUCT"]
sm_line = lineitem["L_SHIPMODE"]

fline = lineitem[
    lq_line.between(4, 36)
    & (si_line == "DELIVER IN PERSON")
    & sm_line.isin(SHIPMODES)
]

# Precompute part columns
pb_part = part["P_BRAND"]
pc_part = part["P_CONTAINER"]
ps_part = part["P_SIZE"]

fpart = part[
    ((pb_part == "Brand#31") & pc_part.isin(SM_SMALL) & ps_part.between(1, 5))
    | ((pb_part == "Brand#43") & pc_part.isin(MED_CONTAINER) & ps_part.between(1, 10))
    | ((pb_part == "Brand#43") & pc_part.isin(LG_CONTAINER) & ps_part.between(1, 15))
]

# Join
df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# Precompute merged columns
lq = df["L_QUANTITY"]
pb = df["P_BRAND"]
pc = df["P_CONTAINER"]

# Final filter (size checks for Brand#31 guaranteed by fpart)
mask_brand31 = (pb == "Brand#31") & lq.between(4, 14)
mask_brand43 = (pb == "Brand#43") & (
    (pc.isin(MED_CONTAINER) & lq.between(15, 25))
    | (pc.isin(LG_CONTAINER)  & lq.between(26, 36))
)
df = df[mask_brand31 | mask_brand43]

# Compute total on GPU
total = (df["L_EXTENDEDPRICE"] * (1.0 - df["L_DISCOUNT"])) .sum()